# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srvnkumr27/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I use regression because `target_next_day_clicks` is a numeric count.

I compare three regression models:

- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor

These models fit the lane because the relationship between recent clicks, impressions, search position, CTR, and next-day clicks may be nonlinear.

I use MAE as the primary metric because it represents the average absolute error in predicted clicks and is easy to interpret.

The Week-4 7-day-average baseline remains the benchmark. The goal is not to reward model complexity, but to check whether the models provide useful improvement over the existing baseline.

In [1]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn matplotlib

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
import os
import duckdb
import pandas as pd
import numpy as np


In [3]:
# Get Hugging Face token securely
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found. Add HF_TOKEN to Colab Secrets."
    )

print("HF_TOKEN found.")

HF_TOKEN found.


In [4]:
# Connect DuckDB to Hugging Face

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("DuckDB connection ready.")

DuckDB connection ready.


In [5]:
MID_PANEL_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

raw_df = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,
        CAST(report_date AS DATE) AS report_date,
        gsc_clicks,
        gsc_impressions,
        gsc_avg_position,
        gsc_data_available
    FROM read_parquet('{MID_PANEL_PATH}')
    WHERE gsc_data_available IS TRUE
    ORDER BY report_date
    """
).df()

print("Raw W05 data shape:", raw_df.shape)

display(raw_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw W05 data shape: (3611061, 7)


,client_hash_id,content_hash_id,report_date,gsc_clicks,gsc_impressions,gsc_avg_position,gsc_data_available
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,0,20,3.350000,True
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,0,1,0.000000,True
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,1,125,4.928000,True
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,0,7,4.000000,True
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,0,11,2.272727,True


In [6]:
raw_df = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,
        CAST(report_date AS DATE) AS report_date,
        gsc_clicks,
        gsc_impressions,
        gsc_avg_position,
        gsc_data_available
    FROM read_parquet('{MID_PANEL_PATH}')
    WHERE gsc_data_available IS TRUE
    ORDER BY report_date
    """
).df()

raw_df["report_date"] = pd.to_datetime(raw_df["report_date"])

print("Raw shape:", raw_df.shape)
display(raw_df.head())

Raw shape: (3611061, 7)


,client_hash_id,content_hash_id,report_date,gsc_clicks,gsc_impressions,gsc_avg_position,gsc_data_available
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,0,20,3.350000,True
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,0,1,0.000000,True
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,1,125,4.928000,True
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,0,7,4.000000,True
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,0,11,2.272727,True


In [7]:
feature_query = f"""
WITH daily_data AS (
    SELECT
        client_hash_id,
        content_hash_id,
        CAST(report_date AS DATE) AS report_date,
        gsc_clicks,
        gsc_impressions,
        gsc_avg_position,

        LEAD(gsc_clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS target_next_day_clicks

    FROM read_parquet('{MID_PANEL_PATH}')

    WHERE gsc_data_available IS TRUE
),

feature_rows AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,

        AVG(gsc_clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS clicks_7d_avg,

        AVG(gsc_impressions) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS impressions_7d_avg,

        AVG(gsc_avg_position) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS position_7d_avg,

        (
            SUM(gsc_clicks) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ) * 1.0
            /
            NULLIF(
                SUM(gsc_impressions) OVER (
                    PARTITION BY client_hash_id, content_hash_id
                    ORDER BY report_date
                    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
                ),
                0
            )
        ) AS ctr_7d,

        CASE
            WHEN DAYOFWEEK(report_date) IN (0, 6)
            THEN 1
            ELSE 0
        END AS is_weekend,

        target_next_day_clicks

    FROM daily_data
)

SELECT *
FROM feature_rows
WHERE target_next_day_clicks IS NOT NULL
ORDER BY report_date
"""

raw_df = con.sql(feature_query).df()

print("Feature dataset shape:", raw_df.shape)

display(raw_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature dataset shape: (3434323, 9)


,client_hash_id,content_hash_id,report_date,clicks_7d_avg,impressions_7d_avg,position_7d_avg,ctr_7d,is_weekend,target_next_day_clicks
0,client_73cda7b4e4f265ea,content_fdac09636baf4f5f,2026-03-01,1.0,6.0,18.833333,0.166667,1,0
1,client_73cda7b4e4f265ea,content_fe4dff4d0fe09f97,2026-03-01,0.0,6.0,8.333333,0.000000,1,0
2,client_73cda7b4e4f265ea,content_ffb2c3d9a3e346f5,2026-03-01,0.0,21.0,2.476190,0.000000,1,0
3,client_795153d5b7850ccf,content_13934a67eb89e470,2026-03-01,0.0,1.0,21.000000,0.000000,1,0
4,client_795153d5b7850ccf,content_4533e1d4135a0561,2026-03-01,0.0,2.0,6.500000,0.000000,1,0


In [8]:
FEATURES = [
    "clicks_7d_avg",
    "impressions_7d_avg",
    "position_7d_avg",
    "ctr_7d",
    "is_weekend"
]

TARGET = "target_next_day_clicks"

print(raw_df.columns.tolist())

missing = [
    col for col in FEATURES + [TARGET]
    if col not in raw_df.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

print("✓ All W05 features are available.")

['client_hash_id', 'content_hash_id', 'report_date', 'clicks_7d_avg', 'impressions_7d_avg', 'position_7d_avg', 'ctr_7d', 'is_weekend', 'target_next_day_clicks']
✓ All W05 features are available.


In [9]:
raw_df["report_date"] = pd.to_datetime(
    raw_df["report_date"]
)

dates = np.sort(
    raw_df["report_date"]
    .dropna()
    .unique()
)

cutoff = dates[
    int(len(dates) * 0.8)
]

train_df = raw_df[
    raw_df["report_date"] < cutoff
].copy()

test_df = raw_df[
    raw_df["report_date"] >= cutoff
].copy()

X_train = train_df[FEATURES].copy()
X_test = test_df[FEATURES].copy()

y_train = train_df[TARGET].astype(float)
y_test = test_df[TARGET].astype(float)

print("Cutoff:", pd.Timestamp(cutoff).date())
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print(
    "Train period:",
    train_df["report_date"].min().date(),
    "to",
    train_df["report_date"].max().date()
)

print(
    "Test period:",
    test_df["report_date"].min().date(),
    "to",
    test_df["report_date"].max().date()
)

Cutoff: 2026-03-25
Train rows: 2713724
Test rows: 720599
Train period: 2026-03-01 to 2026-03-24
Test period: 2026-03-25 to 2026-03-30


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [10]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
models = {
    "Decision Tree": DecisionTreeRegressor(
        max_depth=6,
        min_samples_leaf=30,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=20,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        min_samples_leaf=20,
        random_state=42
    )
}

In [ ]:
trained_models = {}
predictions = {}
results = []

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_test
    )

    # Clicks cannot be negative
    pred = np.clip(
        pred,
        0,
        None
    )

    trained_models[name] = model
    predictions[name] = pred

    mae = mean_absolute_error(
        y_test,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred
        )
    )

    r2 = r2_score(
        y_test,
        pred
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

model_results = pd.DataFrame(results)

display(model_results)

Training Decision Tree...
Training Random Forest...


In [ ]:
comparison = pd.concat(
    [
        pd.DataFrame([{
            "Model": "Week-4 Baseline",
            "MAE": baseline_mae,
            "RMSE": baseline_rmse,
            "R2": baseline_r2
        }]),
        model_results
    ],
    ignore_index=True
)

comparison = (
    comparison
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(
    comparison.style.format({
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "R2": "{:.4f}"
    })
)

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.